# House Prices: repeated-CV feature combination ablation

## tl;dr

동일한 repeated 5-fold × 3 seeds에서 7개 feature 조합을 직접 비교했다.
TotalSF + QualityLivingArea가 CV 0.143928로 baseline 0.145881보다
0.001953 개선되어 1위였고, 세 feature 전체 조합은 0.144413으로 2위였다.
다만 1위도 seed 42에서는 악화되고 seeds 2026·3407에서만 개선되어
강한 확정 결론보다 우선 제출 후보로 해석한다. 예측 blend는 사용하지 않았다.


## Context & Methods

현재 메인 validation-change 기준인 shuffled 5-fold를 split seeds
42, 2026, 3407에서 반복한다. 기준 BaseMLP의 검증된 fold/OOF 결과는
고정 참조하고, feature 후보만 15 folds씩 새로 학습한다.

### Key Assumptions

- 기준 입력은 public 0.12313 제출과 같은 historical BaseMLP recipe이며,
  모델링 복사본의 Id=524, YearRemodAdd=2007 보정을 유지한다.
- 모든 1,460개 train 행을 유지하고 Id는 모델 입력에서 제외한다.
- feature는 target을 쓰지 않는 행 단위 산술식이며 원본 열을 수정하지 않는다.
- 각 후보는 하나의 MLP 입력이다. 모델 예측끼리 blend하지 않는다.
- 주 지표는 동일 15-fold RMSLE 평균과 baseline 대비 paired delta다.
- test 데이터와 Kaggle public score는 학습·선택에 사용하지 않는다.


## Data

### 1. Setup, model, training loop, and train-only input


In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
if not (ROOT / "data" / "train.csv").exists():
    ROOT = ROOT.parent.parent

TRAIN_PATH = ROOT / "data" / "train.csv"
EXPERIMENTS_PATH = ROOT / "reports" / "experiments.csv"
NOTEBOOK_PATH = (
    ROOT / "notebooks" / "feature_engineering" /
    "basemlp_repeated_cv_feature_combinations.ipynb"
)
REPORT_DIR = ROOT / "reports" / "feature_engineering"
ARTIFACT_DIR = ROOT / "artifacts" / "feature_engineering"
RUN_ID = "BASEMLP-20260720-FEAT-COMBOS-REPEATED-NB-12"
FOLD_RESULTS_PATH = REPORT_DIR / "basemlp_repeated_cv_feature_combo_fold_results.csv"
RESULTS_PATH = REPORT_DIR / "basemlp_repeated_cv_feature_combo_results.csv"
OOF_RESULTS_PATH = REPORT_DIR / "basemlp_repeated_cv_feature_combo_oof.csv"
SUMMARY_PATH = REPORT_DIR / "basemlp_repeated_cv_feature_combo_summary.json"
BASELINE_FOLD_RESULTS_PATH = (
    ROOT / "reports" / "validation_test" /
    "basemlp_repeated_cv_baseline_fold_results.csv"
)
BASELINE_OOF_PATH = (
    ROOT / "reports" / "validation_test" /
    "basemlp_repeated_cv_baseline_oof.csv"
)
BASELINE_SUMMARY_PATH = (
    ROOT / "reports" / "validation_test" /
    "basemlp_repeated_cv_baseline_summary.json"
)
BASELINE_EXPERIMENT_ID = "VALBASE-20260720-BASEMLP-REPEATED-3SEED-NB-03"

SPLIT_SEEDS = (42, 2026, 3407)
N_SPLITS = 5
HIDDEN_DIMS = (128, 64)
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
BATCH_SIZE = 64
MAX_EPOCHS = 200
PATIENCE = 25
MIN_DELTA = 1e-5
DEVICE = torch.device("cpu")

NUMERIC_COLUMNS = [
    "LotFrontage", "LotArea", "OverallQual", "OverallCond", "YearBuilt",
    "YearRemodAdd", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea",
    "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath", "BedroomAbvGr",
    "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageYrBlt", "GarageCars",
    "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch",
    "ScreenPorch", "PoolArea", "MiscVal", "YrSold",
]
BASEMENT_COLUMNS = [
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"
]
GARAGE_COLUMNS = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]
class BaseMLP(nn.Module):
    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.hidden1 = nn.Linear(input_dim, HIDDEN_DIMS[0])
        self.hidden2 = nn.Linear(HIDDEN_DIMS[0], HIDDEN_DIMS[1])
        self.output = nn.Linear(HIDDEN_DIMS[1], 1)
        self.activation = nn.ReLU()
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_normal_(self.hidden1.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden1.bias)
        nn.init.kaiming_normal_(self.hidden2.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden2.bias)
        nn.init.xavier_normal_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        hidden = self.activation(self.hidden1(inputs))
        hidden = self.activation(self.hidden2(hidden))
        return self.output(hidden).squeeze(1)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def predict_log_target(
    model: BaseMLP,
    features: np.ndarray,
    target_mean: float,
    target_std: float,
) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        tensor = torch.as_tensor(features, dtype=torch.float32, device=DEVICE)
        standardized = model(tensor).cpu().numpy().astype(np.float64)
    return standardized * target_std + target_mean


def train_fold(
    X_train: np.ndarray,
    y_train_log: np.ndarray,
    X_valid: np.ndarray,
    y_valid_log: np.ndarray,
    seed: int,
    experiment_id: str,
    fold: int,
    checkpoint_path: Path,
    epoch_log_path: Path,
) -> tuple[BaseMLP, dict[str, float | int]]:
    set_seed(seed)
    target_mean = float(y_train_log.mean())
    target_std = float(y_train_log.std(ddof=0))
    y_train_standardized = (y_train_log - target_mean) / target_std
    y_valid_standardized = (y_valid_log - target_mean) / target_std

    train_dataset = TensorDataset(
        torch.as_tensor(X_train, dtype=torch.float32),
        torch.as_tensor(y_train_standardized, dtype=torch.float32),
    )
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )
    X_valid_tensor = torch.as_tensor(X_valid, dtype=torch.float32, device=DEVICE)
    y_valid_tensor = torch.as_tensor(
        y_valid_standardized, dtype=torch.float32, device=DEVICE
    )

    model = BaseMLP(X_train.shape[1]).to(DEVICE)
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    best_score = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    epoch_rows: list[dict[str, float | int]] = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0
        for batch_features, batch_target in train_loader:
            batch_features = batch_features.to(DEVICE)
            batch_target = batch_target.to(DEVICE)
            optimizer.zero_grad()
            prediction = model(batch_features)
            loss = loss_fn(prediction, batch_target)
            loss.backward()
            optimizer.step()
            train_loss_sum += float(loss.detach().cpu()) * len(batch_features)
            train_count += len(batch_features)

        model.eval()
        with torch.no_grad():
            valid_standardized = model(X_valid_tensor)
            valid_loss = loss_fn(valid_standardized, y_valid_tensor)
            valid_log_prediction = (
                valid_standardized.cpu().numpy().astype(np.float64) * target_std
                + target_mean
            )
        valid_rmsle = float(
            np.sqrt(np.mean((y_valid_log - valid_log_prediction) ** 2))
        )
        epoch_rows.append(
            {
                "epoch": epoch,
                "train_loss": train_loss_sum / train_count,
                "validation_loss": float(valid_loss.cpu()),
                "validation_rmsle": valid_rmsle,
                "learning_rate": float(optimizer.param_groups[0]["lr"]),
            }
        )

        if valid_rmsle < best_score - MIN_DELTA:
            best_score = valid_rmsle
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(
                {
                    "experiment_id": experiment_id,
                    "fold": fold,
                    "input_dim": int(X_train.shape[1]),
                    "model_state_dict": model.state_dict(),
                    "target_mean": target_mean,
                    "target_std": target_std,
                    "architecture": [*HIDDEN_DIMS, 1],
                    "seed": seed,
                },
                checkpoint_path,
            )
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            break

    pd.DataFrame(epoch_rows).to_csv(epoch_log_path, index=False)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    restored_prediction = predict_log_target(
        model, X_valid, checkpoint["target_mean"], checkpoint["target_std"]
    )
    restored_rmsle = float(
        np.sqrt(np.mean((y_valid_log - restored_prediction) ** 2))
    )
    if not np.isclose(restored_rmsle, best_score, rtol=0, atol=1e-6):
        raise RuntimeError("Restored checkpoint score does not match the saved best score.")
    return model, {
        "best_epoch": best_epoch,
        "stopping_epoch": epoch,
        "best_validation_rmsle": best_score,
        "restored_validation_rmsle": restored_rmsle,
        "target_mean": target_mean,
        "target_std": target_std,
    }


def append_experiment(row: dict[str, str]) -> None:
    with EXPERIMENTS_PATH.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames
        existing_ids = {existing["experiment_id"] for existing in reader}
    if fieldnames is None:
        raise RuntimeError("reports/experiments.csv has no header.")
    if row["experiment_id"] in existing_ids:
        return
    with EXPERIMENTS_PATH.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writerow({field: row.get(field, "") for field in fieldnames})


train = pd.read_csv(TRAIN_PATH, keep_default_na=False)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
assert train.shape == (1460, 81)
assert train["Id"].is_unique
assert train["SalePrice"].gt(0).all()
categorical_columns = [
    column
    for column in train.columns
    if column not in {*NUMERIC_COLUMNS, "Id", "SalePrice"}
]
assert len(NUMERIC_COLUMNS) == 34
assert len(categorical_columns) == 45

y = train["SalePrice"].to_numpy(dtype=np.float64)
y_log = np.log1p(y)
print(f"train: {train.shape[0]:,} rows × {train.shape[1]} columns")
print(f"train sha256: {sha256(TRAIN_PATH)}")
print("data scope: train.csv only; test/public not loaded")
print("external project script imports: 0")


train: 1,460 rows × 81 columns
train sha256: 1e18addf81e5e4d347cc17ee6075bbe4a42b7fa26b9e5b063e8f692a5f929d41
data scope: train.csv only; test/public not loaded
external project script imports: 0


## Feature construction

### 2. Build and audit all seven feature subsets


In [2]:
def clean_rows_historical_basemlp(
    frame: pd.DataFrame,
    categorical_columns: list[str],
) -> pd.DataFrame:
    cleaned = frame.copy()
    for column in NUMERIC_COLUMNS:
        cleaned[column] = pd.to_numeric(
            cleaned[column].replace({"NA": np.nan, "": np.nan}),
            errors="coerce",
        )

    cleaned.loc[
        cleaned["Id"].eq(524), "YearRemodAdd"
    ] = 2007

    basement_absent = cleaned["TotalBsmtSF"].fillna(0).eq(0)
    garage_absent = (
        cleaned["GarageCars"].fillna(0).eq(0)
        & cleaned["GarageArea"].fillna(0).eq(0)
    )

    for column in BASEMENT_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & basement_absent, column] = "NoBasement"
        cleaned.loc[missing & ~basement_absent, column] = "Unknown"

    for column in GARAGE_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & garage_absent, column] = "NoGarage"
        cleaned.loc[missing & ~garage_absent, column] = "Unknown"
    cleaned.loc[garage_absent, "GarageYrBlt"] = 0.0

    fireplace_absent = cleaned["Fireplaces"].fillna(0).eq(0)
    pool_absent = cleaned["PoolArea"].fillna(0).eq(0)
    cleaned.loc[
        cleaned["FireplaceQu"].isin(["NA", ""]) & fireplace_absent,
        "FireplaceQu",
    ] = "NoFireplace"
    cleaned.loc[
        cleaned["PoolQC"].isin(["NA", ""]) & pool_absent,
        "PoolQC",
    ] = "NoPool"

    for column, label in {
        "Alley": "NoAlley",
        "Fence": "NoFence",
        "MiscFeature": "NoMiscFeature",
    }.items():
        cleaned[column] = cleaned[column].replace({"NA": label, "": label})

    cleaned["MSSubClass"] = cleaned["MSSubClass"].map(
        lambda value: f"SC{value}"
    )
    cleaned["MoSold"] = cleaned["MoSold"].map(
        lambda value: f"M{int(value):02d}"
    )
    for column in categorical_columns:
        cleaned[column] = cleaned[column].replace(
            {"NA": "Unknown", "": "Unknown"}
        )
    return cleaned


base_X = clean_rows_historical_basemlp(
    train.drop(columns="SalePrice"), categorical_columns
)
assert float(
    base_X.loc[base_X["Id"].eq(524), "YearRemodAdd"].iloc[0]
) == 2007
assert base_X.shape == (1460, 80)

FEATURE_SETS = {
    "total_sf": ("TotalSF",),
    "total_bathrooms": ("TotalBathrooms",),
    "quality_living_area": ("QualityLivingArea",),
    "total_sf_total_bathrooms": ("TotalSF", "TotalBathrooms"),
    "total_sf_quality_living_area": ("TotalSF", "QualityLivingArea"),
    "total_bathrooms_quality_living_area": (
        "TotalBathrooms", "QualityLivingArea"
    ),
    "all_three": (
        "TotalSF", "TotalBathrooms", "QualityLivingArea"
    ),
}
FEATURE_DESCRIPTIONS = {
    "TotalSF": "TotalBsmtSF + 1stFlrSF + 2ndFlrSF",
    "TotalBathrooms": (
        "FullBath + 0.5*HalfBath + BsmtFullBath + 0.5*BsmtHalfBath"
    ),
    "QualityLivingArea": "OverallQual * GrLivArea",
}


def add_engineered_features(
    frame: pd.DataFrame,
    feature_names: tuple[str, ...],
) -> pd.DataFrame:
    featured = frame.copy()
    if "TotalSF" in feature_names:
        featured["TotalSF"] = (
            featured[
                ["TotalBsmtSF", "1stFlrSF", "2ndFlrSF"]
            ].fillna(0).sum(axis=1)
        )
    if "TotalBathrooms" in feature_names:
        featured["TotalBathrooms"] = (
            featured["FullBath"].fillna(0)
            + 0.5 * featured["HalfBath"].fillna(0)
            + featured["BsmtFullBath"].fillna(0)
            + 0.5 * featured["BsmtHalfBath"].fillna(0)
        )
    if "QualityLivingArea" in feature_names:
        featured["QualityLivingArea"] = (
            featured["OverallQual"] * featured["GrLivArea"]
        )
    return featured


feature_frames: dict[str, pd.DataFrame] = {}
audit_rows: list[dict[str, object]] = []
for candidate, feature_names in FEATURE_SETS.items():
    featured = add_engineered_features(base_X, feature_names)
    assert featured.drop(columns=list(feature_names)).equals(base_X)
    assert len(featured) == len(base_X)
    for feature_name in feature_names:
        assert featured[feature_name].notna().all()
        assert np.isfinite(featured[feature_name]).all()
        assert featured[feature_name].ge(0).all()
        assert featured[feature_name].nunique() > 1
    feature_frames[candidate] = featured
    audit_rows.append({
        "candidate": candidate,
        "features": " + ".join(feature_names),
        "generated_feature_count": len(feature_names),
        "rows": len(featured),
        "original_columns_modified": 0,
        "row_specific_rules_added": 0,
    })

baseline_fold_results = pd.read_csv(
    BASELINE_FOLD_RESULTS_PATH
).sort_values(["split_seed", "fold"]).reset_index(drop=True)
baseline_oof = pd.read_csv(BASELINE_OOF_PATH)
baseline_summary = json.loads(
    BASELINE_SUMMARY_PATH.read_text(encoding="utf-8")
)
assert len(baseline_fold_results) == 15
assert baseline_fold_results["split_seed"].tolist() == [
    seed for seed in SPLIT_SEEDS for _ in range(N_SPLITS)
]
assert baseline_oof["Id"].equals(train["Id"])
assert np.isclose(
    baseline_summary["results"]["cv_mean"],
    0.14588108497354113,
    rtol=0,
    atol=1e-12,
)
assert (
    baseline_summary["recipe"]["id_524_yearremodadd"] == 2007
)
display(pd.DataFrame(audit_rows))
print(
    "Frozen reference loaded:",
    BASELINE_EXPERIMENT_ID,
    f"CV={baseline_summary['results']['cv_mean']:.6f}",
)

def make_preprocessor(numeric_columns: list[str]) -> ColumnTransformer:
    numeric = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical = Pipeline(
        [
            (
                "imputer",
                SimpleImputer(strategy="constant", fill_value="Unknown"),
            ),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", numeric, numeric_columns),
            ("categorical", categorical, categorical_columns),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )

,candidate,features,generated_feature_count,rows,original_columns_modified,row_specific_rules_added
0,total_sf,TotalSF,1,1460,0,0
1,total_bathrooms,TotalBathrooms,1,1460,0,0
2,quality_living_area,QualityLivingArea,1,1460,0,0
3,total_sf_total_bathrooms,TotalSF + TotalBathrooms,2,1460,0,0
4,total_sf_quality_living_area,TotalSF + QualityLivingArea,2,1460,0,0
5,total_bathrooms_quality_living_area,TotalBathrooms + QualityLivingArea,2,1460,0,0
6,all_three,TotalSF + TotalBathrooms + QualityLivingArea,3,1460,0,0


Frozen reference loaded: VALBASE-20260720-BASEMLP-REPEATED-3SEED-NB-03 CV=0.145881


## Repeated model fitting

### 3. Train seven candidates on identical 15 folds


In [3]:
OUTPUT_DIR = ARTIFACT_DIR / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

candidate_oof_by_seed: dict[str, np.ndarray] = {}
fold_rows: list[dict[str, object]] = []
started = time.perf_counter()

for candidate_index, (candidate, feature_names) in enumerate(
    FEATURE_SETS.items(), start=1
):
    candidate_started = time.perf_counter()
    candidate_experiment_id = f"{RUN_ID}-{candidate.upper()}"
    candidate_dir = OUTPUT_DIR / candidate
    checkpoint_dir = candidate_dir / "checkpoints"
    preprocessor_dir = candidate_dir / "preprocessors"
    epoch_log_dir = candidate_dir / "logs"
    for directory in (
        checkpoint_dir, preprocessor_dir, epoch_log_dir
    ):
        directory.mkdir(parents=True, exist_ok=True)

    candidate_frame = feature_frames[candidate]
    candidate_numeric_columns = [
        *NUMERIC_COLUMNS, *feature_names
    ]
    candidate_oof = np.full(
        (len(SPLIT_SEEDS), len(train)),
        np.nan,
        dtype=np.float64,
    )

    for seed_index, split_seed in enumerate(SPLIT_SEEDS):
        splitter = KFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=split_seed,
        )
        for fold, (train_index, valid_index) in enumerate(
            splitter.split(candidate_frame), start=1
        ):
            split_number = seed_index * N_SPLITS + fold
            model_seed = split_seed + fold
            checkpoint_path = (
                checkpoint_dir /
                f"seed_{split_seed}_fold_{fold}_best.pt"
            )
            preprocessor_path = (
                preprocessor_dir /
                f"seed_{split_seed}_fold_{fold}_preprocessor.joblib"
            )
            epoch_log_path = (
                epoch_log_dir /
                f"seed_{split_seed}_fold_{fold}_epochs.csv"
            )

            preprocessor = make_preprocessor(
                candidate_numeric_columns
            )
            X_train = preprocessor.fit_transform(
                candidate_frame.iloc[train_index]
            ).astype(np.float32)
            X_valid = preprocessor.transform(
                candidate_frame.iloc[valid_index]
            ).astype(np.float32)
            joblib.dump(preprocessor, preprocessor_path)

            model, training_result = train_fold(
                X_train,
                y_log[train_index],
                X_valid,
                y_log[valid_index],
                seed=model_seed,
                experiment_id=candidate_experiment_id,
                fold=split_number,
                checkpoint_path=checkpoint_path,
                epoch_log_path=epoch_log_path,
            )
            valid_log_prediction = predict_log_target(
                model,
                X_valid,
                training_result["target_mean"],
                training_result["target_std"],
            )
            candidate_oof[seed_index, valid_index] = (
                valid_log_prediction
            )
            baseline_rmsle = float(
                baseline_fold_results.loc[
                    (
                        baseline_fold_results["split_seed"].eq(
                            split_seed
                        )
                        & baseline_fold_results["fold"].eq(fold)
                    ),
                    "rmsle",
                ].iloc[0]
            )
            restored_rmsle = training_result[
                "restored_validation_rmsle"
            ]
            fold_rows.append({
                "candidate": candidate,
                "features": " + ".join(feature_names),
                "split_seed": split_seed,
                "fold": fold,
                "split_number": split_number,
                "model_seed": model_seed,
                "training_rows": len(train_index),
                "validation_rows": len(valid_index),
                "encoded_features": X_train.shape[1],
                "best_epoch": training_result["best_epoch"],
                "stopping_epoch": training_result[
                    "stopping_epoch"
                ],
                "rmsle": restored_rmsle,
                "baseline_rmsle": baseline_rmsle,
                "paired_delta": restored_rmsle - baseline_rmsle,
                "checkpoint_path": str(
                    checkpoint_path.relative_to(ROOT)
                ),
                "preprocessor_path": str(
                    preprocessor_path.relative_to(ROOT)
                ),
                "epoch_log_path": str(
                    epoch_log_path.relative_to(ROOT)
                ),
            })

            del model, preprocessor, X_train, X_valid

    assert not np.isnan(candidate_oof).any()
    candidate_oof_by_seed[candidate] = candidate_oof
    candidate_rows = [
        row for row in fold_rows
        if row["candidate"] == candidate
    ]
    candidate_scores = np.array(
        [row["rmsle"] for row in candidate_rows],
        dtype=np.float64,
    )
    candidate_delta = np.array(
        [row["paired_delta"] for row in candidate_rows],
        dtype=np.float64,
    )
    print(
        f"[{candidate_index}/{len(FEATURE_SETS)}] {candidate}: "
        f"CV={candidate_scores.mean():.6f}, "
        f"delta={candidate_delta.mean():+.6f}, "
        f"improved={int((candidate_delta < 0).sum())}/15, "
        f"elapsed={time.perf_counter() - candidate_started:.1f}s"
    )

fold_results = pd.DataFrame(fold_rows)
fold_results.to_csv(FOLD_RESULTS_PATH, index=False)
duration_seconds = time.perf_counter() - started
assert len(fold_results) == len(FEATURE_SETS) * len(SPLIT_SEEDS) * N_SPLITS
print(f"All 105 candidate folds completed in {duration_seconds:.2f} seconds")


[1/7] total_sf: CV=0.145864, delta=-0.000017, improved=6/15, elapsed=7.5s


[2/7] total_bathrooms: CV=0.147640, delta=+0.001759, improved=6/15, elapsed=6.9s


[3/7] quality_living_area: CV=0.145101, delta=-0.000780, improved=6/15, elapsed=6.7s


[4/7] total_sf_total_bathrooms: CV=0.145080, delta=-0.000801, improved=7/15, elapsed=6.4s


[5/7] total_sf_quality_living_area: CV=0.143928, delta=-0.001953, improved=7/15, elapsed=6.4s


[6/7] total_bathrooms_quality_living_area: CV=0.145495, delta=-0.000387, improved=7/15, elapsed=6.7s


[7/7] all_three: CV=0.144413, delta=-0.001468, improved=8/15, elapsed=8.3s
All 105 candidate folds completed in 48.96 seconds


## Results

### 4. Rank candidates with paired and OOF checks


In [4]:
baseline_cv_mean = float(
    baseline_summary["results"]["cv_mean"]
)
baseline_cv_std = float(
    baseline_summary["results"]["cv_std"]
)
baseline_repeated_oof = float(
    baseline_summary["results"]["repeated_oof_rmsle"]
)
baseline_ensemble_oof = float(
    baseline_summary["results"][
        "three_seed_ensemble_oof_rmsle"
    ]
)

result_rows: list[dict[str, object]] = []
seed_delta_rows: list[dict[str, object]] = []
oof_results = pd.DataFrame({
    "Id": train["Id"].to_numpy(dtype=np.int64),
    "actual_log1p": y_log,
})

for candidate, feature_names in FEATURE_SETS.items():
    candidate_folds = (
        fold_results.loc[fold_results["candidate"].eq(candidate)]
        .sort_values(["split_seed", "fold"])
        .reset_index(drop=True)
    )
    scores = candidate_folds["rmsle"].to_numpy(dtype=np.float64)
    paired_delta = candidate_folds["paired_delta"].to_numpy(
        dtype=np.float64
    )
    candidate_oof = candidate_oof_by_seed[candidate]

    seed_oof_scores = {}
    seed_cv_deltas = {}
    for seed_index, split_seed in enumerate(SPLIT_SEEDS):
        seed_prediction = candidate_oof[seed_index]
        seed_oof_scores[str(split_seed)] = float(np.sqrt(np.mean(
            (y_log - seed_prediction) ** 2
        )))
        candidate_seed_cv = float(
            candidate_folds.loc[
                candidate_folds["split_seed"].eq(split_seed),
                "rmsle",
            ].mean()
        )
        baseline_seed_cv = float(
            baseline_fold_results.loc[
                baseline_fold_results["split_seed"].eq(split_seed),
                "rmsle",
            ].mean()
        )
        seed_cv_deltas[str(split_seed)] = (
            candidate_seed_cv - baseline_seed_cv
        )
        seed_delta_rows.append({
            "candidate": candidate,
            "split_seed": split_seed,
            "candidate_cv": candidate_seed_cv,
            "baseline_cv": baseline_seed_cv,
            "paired_delta": (
                candidate_seed_cv - baseline_seed_cv
            ),
        })
        oof_results[
            f"{candidate}__seed_{split_seed}_oof_log"
        ] = seed_prediction

    repeated_oof = float(np.sqrt(np.mean(
        (
            np.tile(y_log, (len(SPLIT_SEEDS), 1))
            - candidate_oof
        ) ** 2
    )))
    ensemble_oof_prediction = candidate_oof.mean(axis=0)
    ensemble_oof = float(np.sqrt(np.mean(
        (y_log - ensemble_oof_prediction) ** 2
    )))
    oof_results[
        f"{candidate}__three_seed_ensemble_oof_log"
    ] = ensemble_oof_prediction

    cv_delta = float(scores.mean() - baseline_cv_mean)
    repeated_oof_delta = float(
        repeated_oof - baseline_repeated_oof
    )
    ensemble_oof_delta = float(
        ensemble_oof - baseline_ensemble_oof
    )
    improved_seeds = int(sum(
        delta < 0 for delta in seed_cv_deltas.values()
    ))
    if (
        cv_delta <= -0.0005
        and repeated_oof_delta < 0
        and ensemble_oof_delta < 0
        and improved_seeds >= 2
    ):
        decision = "keep"
    elif abs(cv_delta) < 0.0005:
        decision = "tie"
    else:
        decision = "reject"

    result_rows.append({
        "candidate": candidate,
        "features": " + ".join(feature_names),
        "feature_count": len(feature_names),
        "cv_mean": float(scores.mean()),
        "cv_std": float(scores.std(ddof=1)),
        "cv_delta_vs_baseline": cv_delta,
        "paired_delta_std": float(paired_delta.std(ddof=1)),
        "improved_folds": int((paired_delta < 0).sum()),
        "improved_seeds": improved_seeds,
        "repeated_oof_rmsle": repeated_oof,
        "repeated_oof_delta_vs_baseline": repeated_oof_delta,
        "three_seed_ensemble_oof_rmsle": ensemble_oof,
        "ensemble_oof_delta_vs_baseline": ensemble_oof_delta,
        "seed_42_cv_delta": seed_cv_deltas["42"],
        "seed_2026_cv_delta": seed_cv_deltas["2026"],
        "seed_3407_cv_delta": seed_cv_deltas["3407"],
        "decision": decision,
    })

results = (
    pd.DataFrame(result_rows)
    .sort_values(["cv_mean", "three_seed_ensemble_oof_rmsle"])
    .reset_index(drop=True)
)
results.insert(0, "rank", np.arange(1, len(results) + 1))
seed_deltas = pd.DataFrame(seed_delta_rows)
results.to_csv(RESULTS_PATH, index=False)
oof_results.to_csv(OOF_RESULTS_PATH, index=False)

artifact_checks = {
    "checkpoint_count": len(list(
        OUTPUT_DIR.glob("*/checkpoints/*.pt")
    )),
    "preprocessor_count": len(list(
        OUTPUT_DIR.glob("*/preprocessors/*.joblib")
    )),
    "epoch_log_count": len(list(
        OUTPUT_DIR.glob("*/logs/*.csv")
    )),
}
expected_artifact_count = (
    len(FEATURE_SETS) * len(SPLIT_SEEDS) * N_SPLITS
)
assert artifact_checks == {
    "checkpoint_count": expected_artifact_count,
    "preprocessor_count": expected_artifact_count,
    "epoch_log_count": expected_artifact_count,
}
assert np.isfinite(
    results.select_dtypes(include=[np.number]).to_numpy()
).all()
assert results["candidate"].nunique() == 7
assert oof_results["Id"].equals(train["Id"])

run_timestamp = datetime.now(timezone.utc).isoformat()
summary = {
    "run_timestamp_utc": run_timestamp,
    "run_id": RUN_ID,
    "objective": (
        "Compare all seven non-empty combinations of TotalSF, "
        "TotalBathrooms, and QualityLivingArea under frozen repeated CV"
    ),
    "reference": {
        "experiment_id": BASELINE_EXPERIMENT_ID,
        "cv_mean": baseline_cv_mean,
        "cv_std": baseline_cv_std,
        "repeated_oof_rmsle": baseline_repeated_oof,
        "three_seed_ensemble_oof_rmsle": baseline_ensemble_oof,
        "id_524_yearremodadd": 2007,
    },
    "method": {
        "splitter": "KFold(5, shuffle=True)",
        "split_seeds": list(SPLIT_SEEDS),
        "model_seed": "split_seed + fold",
        "feature_formulas": FEATURE_DESCRIPTIONS,
        "candidate_count": len(FEATURE_SETS),
        "candidate_models_trained": expected_artifact_count,
        "prediction_blending": False,
        "test_data_loaded": False,
        "public_scores_loaded": False,
        "small_delta_rule": (
            "absolute primary CV delta below 0.0005 is tied"
        ),
    },
    "results": json.loads(
        results.to_json(orient="records")
    ),
    "seed_deltas": json.loads(
        seed_deltas.to_json(orient="records")
    ),
    "duration_seconds": duration_seconds,
    "validation": {
        **artifact_checks,
        "candidate_fold_rows": len(fold_results),
        "oof_rows": len(oof_results),
        "all_original_rows_retained": True,
        "original_columns_modified": 0,
        "external_project_script_imports": 0,
    },
    "artifacts": {
        "notebook": str(NOTEBOOK_PATH.relative_to(ROOT)),
        "fold_results": str(FOLD_RESULTS_PATH.relative_to(ROOT)),
        "ranked_results": str(RESULTS_PATH.relative_to(ROOT)),
        "oof_predictions": str(OOF_RESULTS_PATH.relative_to(ROOT)),
        "summary": str(SUMMARY_PATH.relative_to(ROOT)),
        "model_artifacts": str(OUTPUT_DIR.relative_to(ROOT)),
    },
    "caveats": [
        (
            "Seven candidates are ranked on repeated folds over the same "
            "1,460 training rows; selection optimism remains."
        ),
        (
            "Repeated split seeds reduce fold luck but are not independent "
            "test data."
        ),
        (
            "Leaderboard behavior cannot be inferred from small local "
            "differences; no test or public score entered this experiment."
        ),
    ],
}
SUMMARY_PATH.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

for row in results.to_dict(orient="records"):
    candidate = row["candidate"]
    feature_names = FEATURE_SETS[candidate]
    candidate_folds = fold_results.loc[
        fold_results["candidate"].eq(candidate)
    ].sort_values(["split_seed", "fold"])
    experiment_id = f"{RUN_ID}-{candidate.upper()}"
    append_experiment({
        "experiment_id": experiment_id,
        "datetime": run_timestamp,
        "objective": (
            "Evaluate one BaseMLP with generated features: "
            + " + ".join(feature_names)
        ),
        "data_version": f"train sha256={sha256(TRAIN_PATH)}",
        "split_strategy": (
            "KFold(5,shuffle=True) repeated at seeds 42|2026|3407"
        ),
        "folds": "5x3",
        "seed": "42|2026|3407; model seed=split_seed+fold",
        "metric": "RMSLE / RMSE on log1p target",
        "preprocessing": (
            "Historical BaseMLP fold median/scaling/one-hot; "
            "Id=524 YearRemodAdd=2007 modeling-copy correction; "
            "all rows retained; train only"
        ),
        "features": (
            "79 original predictors + "
            + " + ".join(feature_names)
            + "; Id excluded"
        ),
        "model": "Manual PyTorch BaseMLP; no prediction blend",
        "architecture": (
            "input->128(ReLU)->64(ReLU)->1; "
            "Kaiming hidden/Xavier output"
        ),
        "optimizer": "Adam(weight_decay=0)",
        "loss": (
            "MSELoss on fold-standardized log1p(SalePrice)"
        ),
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "best_epoch": json.dumps(
            candidate_folds["best_epoch"].astype(int).tolist()
        ),
        "hyperparameters": json.dumps({
            "reference_experiment": BASELINE_EXPERIMENT_ID,
            "generated_features": list(feature_names),
            "feature_formulas": {
                name: FEATURE_DESCRIPTIONS[name]
                for name in feature_names
            },
            "split_seeds": list(SPLIT_SEEDS),
            "prediction_blending": False,
            "test_public_loaded": False,
            "small_delta_rule": 0.0005,
        }),
        "cv_mean": row["cv_mean"],
        "cv_std": row["cv_std"],
        "holdout_score": row[
            "three_seed_ensemble_oof_rmsle"
        ],
        "checkpoint_path": str(
            (OUTPUT_DIR / candidate / "checkpoints").relative_to(ROOT)
        ),
        "artifact_path": " | ".join([
            str(NOTEBOOK_PATH.relative_to(ROOT)),
            str(FOLD_RESULTS_PATH.relative_to(ROOT)),
            str(RESULTS_PATH.relative_to(ROOT)),
            str(OOF_RESULTS_PATH.relative_to(ROOT)),
            str(SUMMARY_PATH.relative_to(ROOT)),
        ]),
        "result": row["decision"],
        "interpretation": (
            f"CV delta vs frozen baseline={row['cv_delta_vs_baseline']:+.6f}; "
            f"repeated OOF delta={row['repeated_oof_delta_vs_baseline']:+.6f}; "
            f"3-seed ensemble OOF delta={row['ensemble_oof_delta_vs_baseline']:+.6f}; "
            f"improved folds={int(row['improved_folds'])}/15; "
            f"improved seeds={int(row['improved_seeds'])}/3."
        ),
        "next_action": (
            "Consider a submission only for a robustly retained candidate; "
            "do not select on public leaderboard."
        ),
    })

display_columns = [
    "rank", "features", "cv_mean", "cv_std",
    "cv_delta_vs_baseline", "improved_folds", "improved_seeds",
    "repeated_oof_rmsle", "repeated_oof_delta_vs_baseline",
    "three_seed_ensemble_oof_rmsle",
    "ensemble_oof_delta_vs_baseline", "decision",
]
display(results[display_columns])
display(seed_deltas.pivot(
    index="candidate",
    columns="split_seed",
    values="paired_delta",
).loc[results["candidate"]])
print(
    f"Baseline: CV={baseline_cv_mean:.6f}, "
    f"repeated OOF={baseline_repeated_oof:.6f}, "
    f"3-seed ensemble OOF={baseline_ensemble_oof:.6f}"
)
print(f"Summary: {SUMMARY_PATH.relative_to(ROOT)}")


,rank,features,cv_mean,cv_std,cv_delta_vs_baseline,improved_folds,improved_seeds,repeated_oof_rmsle,repeated_oof_delta_vs_baseline,three_seed_ensemble_oof_rmsle,ensemble_oof_delta_vs_baseline,decision
0,1,TotalSF + QualityLivingArea,0.143928,0.031134,-0.001953,7,2,0.147037,-0.001152,0.131811,-0.002422,keep
1,2,TotalSF + TotalBathrooms + QualityLivingArea,0.144413,0.024417,-0.001468,8,2,0.146327,-0.001862,0.132963,-0.001270,keep
2,3,TotalSF + TotalBathrooms,0.145080,0.030716,-0.000801,7,1,0.148084,-0.000105,0.133462,-0.000770,reject
3,4,QualityLivingArea,0.145101,0.026525,-0.000780,6,2,0.147346,-0.000843,0.132678,-0.001554,keep
4,5,TotalBathrooms + QualityLivingArea,0.145495,0.030786,-0.000387,7,2,0.148503,0.000314,0.132530,-0.001703,tie
5,6,TotalSF,0.145864,0.027090,-0.000017,6,1,0.148194,0.000005,0.134237,0.000005,tie
6,7,TotalBathrooms,0.147640,0.028110,0.001759,6,1,0.150117,0.001928,0.136215,0.001983,reject


split_seed,42,2026,3407
candidate,,,
total_sf_quality_living_area,0.006509,-0.010721,-0.001646
all_three,-0.002385,-0.003904,0.001884
total_sf_total_bathrooms,0.006962,-0.009397,0.000033
quality_living_area,0.006270,-0.008101,-0.000509
total_bathrooms_quality_living_area,0.008036,-0.007837,-0.001358
total_sf,0.007848,-0.008209,0.000311
total_bathrooms,0.010173,-0.006748,0.001851


Baseline: CV=0.145881, repeated OOF=0.148189, 3-seed ensemble OOF=0.134232
Summary: reports/feature_engineering/basemlp_repeated_cv_feature_combo_summary.json


## Takeaways

- 1위 TotalSF + QualityLivingArea: CV 0.143928, baseline delta -0.001953,
  repeated OOF delta -0.001152, 3-seed ensemble OOF delta -0.002422.
- 2위 전체 세 feature: CV 0.144413, baseline delta -0.001468이며 8/15 folds와
  2/3 seeds에서 개선됐다. 1위와의 CV 차이는 0.000485로 동률 경계 안이다.
- QualityLivingArea 단독은 0.145101로 소폭 개선, TotalSF 단독은 사실상 동률,
  TotalBathrooms 단독은 0.147640으로 악화됐다.
- 후보 대부분이 seed 2026에서 좋아지고 seed 42에서 나빠져 split/model-seed
  민감도가 크다. 7개 후보 선택에 따른 optimism도 남는다.
- 이 노트북은 validation 전용이다. test inference나 submission은 생성하지 않는다.
